In [4]:
# ==========================================================
# 🔥 Resume Screening System (Exact Version For Your File)
# File name: resume.csv.zip
# Inside ZIP: Resume/Resume.csv
# ==========================================================

# If running first time:
# !pip install pandas numpy scikit-learn nltk

import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import zipfile

print("🔄 Starting Resume Screening System...")

# Download stopwords
nltk.download('stopwords', quiet=True)

# ==========================================================
# 1️⃣ LOAD DATASET FROM ZIP (CASE-SENSITIVE)
# ==========================================================

zip_path = "Resume.csv.zip"   # your file name

try:
    with zipfile.ZipFile(zip_path) as z:
        # IMPORTANT: Case sensitive path inside zip
        with z.open("Resume/Resume.csv") as f:
            df = pd.read_csv(f)
    print("✅ Dataset loaded successfully!")
except Exception as e:
    raise Exception(f"❌ Error loading file: {e}")

print("Total resumes:", len(df))
print("Available columns:", df.columns.tolist())

# ==========================================================
# 2️⃣ FIND RESUME TEXT COLUMN
# ==========================================================

if 'Resume_str' in df.columns:
    resume_column = 'Resume_str'
elif 'resume_str' in df.columns:
    resume_column = 'resume_str'
else:
    raise ValueError("❌ Resume text column not found in dataset!")

print("📄 Using resume column:", resume_column)

df = df.dropna(subset=[resume_column])

# ==========================================================
# 3️⃣ CLEAN TEXT
# ==========================================================

stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return " ".join(words)

print("🧹 Cleaning resume text...")
df["cleaned_resume"] = df[resume_column].apply(clean_text)

# ==========================================================
# 4️⃣ DEFINE JOB DESCRIPTION
# ==========================================================

job_description = """
Looking for a Machine Learning Engineer with strong skills in Python,
SQL, Deep Learning, NLP, and Data Analysis.
Experience in building ML models is required.
"""

clean_job = clean_text(job_description)

# ==========================================================
# 5️⃣ TF-IDF + COSINE SIMILARITY
# ==========================================================

print("📊 Calculating similarity scores...")

documents = [clean_job] + df["cleaned_resume"].tolist()

vectorizer = TfidfVectorizer(max_features=5000)
tfidf_matrix = vectorizer.fit_transform(documents)

similarity_scores = cosine_similarity(
    tfidf_matrix[0:1],
    tfidf_matrix[1:]
).flatten()

df["similarity_score"] = similarity_scores

ranked_df = df.sort_values(by="similarity_score", ascending=False)

# ==========================================================
# 6️⃣ SKILL MATCHING
# ==========================================================

skill_list = [
    "python", "machine learning", "deep learning",
    "nlp", "sql", "data analysis",
    "tensorflow", "keras", "pandas", "numpy"
]

def extract_skills(text):
    return [skill for skill in skill_list if skill in text]

def missing_skills(found):
    return list(set(skill_list) - set(found))

ranked_df["matched_skills"] = ranked_df["cleaned_resume"].apply(extract_skills)
ranked_df["missing_skills"] = ranked_df["matched_skills"].apply(missing_skills)

# ==========================================================
# 7️⃣ SHOW TOP 10 CANDIDATES
# ==========================================================

print("\n🏆 TOP 10 CANDIDATES\n")

top_10 = ranked_df.head(10)

for index, row in top_10.iterrows():
    print("Category:", row.get("Category", "N/A"))
    print("Similarity Score:", round(row["similarity_score"], 4))
    print("Matched Skills:", row["matched_skills"])
    print("Missing Skills:", row["missing_skills"])
    print("-" * 60)

# ==========================================================
# 8️⃣ SAVE OUTPUT FILE
# ==========================================================

top_10.to_csv("ranked_resumes_output.csv", index=False)

print("\n✅ SYSTEM COMPLETED SUCCESSFULLY!")
print("📁 Output saved as 'ranked_resumes_output.csv'")

🔄 Starting Resume Screening System...
✅ Dataset loaded successfully!
Total resumes: 2484
Available columns: ['ID', 'Resume_str', 'Resume_html', 'Category']
📄 Using resume column: Resume_str
🧹 Cleaning resume text...
📊 Calculating similarity scores...

🏆 TOP 10 CANDIDATES

Category: ENGINEERING
Similarity Score: 0.2565
Matched Skills: ['python', 'machine learning', 'sql', 'data analysis', 'pandas']
Missing Skills: ['numpy', 'keras', 'deep learning', 'nlp', 'tensorflow']
------------------------------------------------------------
Category: BANKING
Similarity Score: 0.2252
Matched Skills: ['python', 'machine learning', 'sql']
Missing Skills: ['numpy', 'keras', 'pandas', 'deep learning', 'nlp', 'data analysis', 'tensorflow']
------------------------------------------------------------
Category: CONSULTANT
Similarity Score: 0.2183
Matched Skills: ['python', 'machine learning', 'sql', 'data analysis']
Missing Skills: ['numpy', 'keras', 'pandas', 'deep learning', 'nlp', 'tensorflow']
-------